This notebook:
- 9 rolling folds 550 days train, 90 valid
- Trains a primary model using the above
- Based on permutation importance it selects the non-zero features and logs them in top_features_list
- Permutation importance is optimized using custom scorer (profitability)

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression 
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
import shap 
import catboost as cat 
import xgboost as xgb
import lightgbm as lgb
from datetime import timedelta, datetime
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('SE5.csv')
df['Time'] = pd.to_datetime(df['Time'])
df = df.sort_values(by='Time', ascending=True)

In [2]:
train_val_df = df[df["Time"] < "2024-01-01"]
test_df = df[df["Time"] >= "2024-01-01"]

print(f"Train/Validation data shape: {train_val_df.shape}")
print(f"Test data shape: {test_df.shape}")

Train/Validation data shape: (40884, 717)
Test data shape: (2313, 717)


In [3]:
# Helper Functions
def calculate_stability_score(y_true, pred, class1_proportion):
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    if fp == 0:
        return 1e9 if tp > 0 else 0.0
    if class1_proportion == 0:
        return 0.0
    stability = (tp / fp) / class1_proportion
    return stability

def evaluate_model(y_true, y_prob, model_name, class1_proportion_val, thresholds=None):
    """
    Simplified evaluation function with built-in optimal threshold finding,
    with a filter to avoid selecting unrealistic 100% precision/recall.
    Input: True values, probability predictions, profitable trades proportion, explicit thresholds (optional)
    Returns metrics dictionary.
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 100) # a range of 100 threshold option
        
    thresholds = [t for t in thresholds if 0.001 < t < 0.999] # sanity check
    results = []
    
    # Find optimal threshold
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        
        # Get confusion matrix components
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        
        # Calculate evaluation metrics
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        stability = calculate_stability_score(y_true, y_pred, class1_proportion_val)
        
        # Saving metrics and confusion matrics components for later filtering
        results.append({
            'threshold': threshold,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'stability': stability,
            'tn': tn,
            'fp': fp,
            'fn': fn,
            'tp': tp,
            'score': (tp * 800) - (fp * 200) - (fn * 200),
            'score_1': (tp * 800) - (fp * 200),
        })

    if not results:  # Fallback if no thresholds
        # This part remains as a safety fallback
        threshold = 0.5
        y_pred = (y_prob >= threshold).astype(int)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        stability = calculate_stability_score(y_true, y_pred, class1_proportion_val)
        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel()
        score = (tp * 800) - (fp * 200) - (fn * 200)
        score_1 = (tp * 800) - (fp * 200)
    else:
        results_df = pd.DataFrame(results)

        # This to consider options where false positive, false negative and true negative > 10 instances otherwise it's misleading.
        misleading_options = (results_df['fp'] < 10) | (results_df['fn'] < 10) | (results_df['tn'] < 10)
        
        filtered_df = results_df[~(misleading_options)].copy()

        # Fallback if filtering removes all candidates (if all predictions are extreme), use original results if all are deemed extreme
        if filtered_df.empty:
            print(f"Warning: All candidates filtered out for {model_name}. Reverting to original selection without extreme filtering.")
            filtered_df = results_df.copy()
        else:
            print(f"Extreme filtering is done, original results {results_df.shape[0]} row, filtered {filtered_df.shape[0]}")

        # Find best threshold from the filtered (or original) candidates
        # If more than instance have the same max F1, we take the higher f1
        max_score_1 = filtered_df['score_1'].max()
        best_candidates = filtered_df[filtered_df['score_1'] == max_score_1]
        best_row = best_candidates.loc[best_candidates['f1'].idxmax()]
        
        threshold = best_row['threshold']
        precision = best_row['precision']
        recall = best_row['recall']
        f1 = best_row['f1']
        stability = best_row['stability']
        score = best_row['score']
        score_1 = best_row['score_1']
        
        # Re-calculate confusion matrix for the final selected optimal threshold
        y_pred = (y_prob >= threshold).astype(int)
        cm = confusion_matrix(y_true, y_pred)
        
    return {
        'model': model_name,
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'stability': stability,
        'confusion_matrix': cm,
        'score': score,
        'score_1': score_1
    }

In [4]:
from sklearn.metrics import make_scorer

def profitability_scorer(estimator, X, y):
    y_pred = estimator.predict(X)
    TP = ((y == 1) & (y_pred == 1)).sum()
    FP = ((y == 0) & (y_pred == 1)).sum()
    return TP * 800 - FP * 200

custom_scorer = make_scorer(profitability_scorer, greater_is_better=True)

In [ ]:
# Rolling Window Cross-Validation
train_size_days = 550
valid_size_days = 90

fold = 1 
start_date_cv = train_val_df['Time'].min()

top_features_list = list()

while True:
    train_start = start_date_cv
    train_end = train_start + timedelta(days=train_size_days)
    val_start = train_end + timedelta(minutes=5)
    val_end = val_start + timedelta(days=valid_size_days)
    if val_end > train_val_df['Time'].max():
        break
    train_set = train_val_df[(train_val_df['Time']>=train_start) & (train_val_df['Time']<train_end)]
    valid_set = train_val_df[(train_val_df['Time']>=val_start) & (train_val_df['Time']<val_end)]
    x_tr_raw, y_tr = train_set.drop(columns=['Time','class']), train_set['class']
    x_va_raw, y_va = valid_set.drop(columns=['Time','class']), valid_set['class']
    class_weight = y_tr.value_counts()[0]/y_tr.value_counts()[1]
    
    n_features = x_tr_raw.shape[1]
    print(f"Fold {fold}")
    print(f"Training Period: {train_start.strftime('%Y-%m-%d')} - {train_end.strftime('%Y-%m-%d')}")
    print(f"Validation Period: {val_start.strftime('%Y-%m-%d')} - {val_end.strftime('%Y-%m-%d')}")
    
    # Catboost
    # training temp model for permutation importance
    # tmp_model = cat.CatBoostClassifier(random_state=1, scale_pos_weight=class_weight, loss_function='Logloss', eval_metric='F1', early_stopping_rounds=5, verbose=0)
    # tmp_model.fit(x_tr_raw, y_tr, eval_set=[(x_va_raw, y_va)])

    # LogReg
    # tmp_model = LogisticRegression(max_iter=1000, C=0.01, class_weight={0:1, 1: class_weight}, random_state=1)
    # logreg_scaler = StandardScaler()
    # x_tr_raw_scaled = logreg_scaler.fit_transform(x_tr_raw)
    # tmp_model.fit(x_tr_raw_scaled, y_tr)

    # XGBoost
    # tmp_model = xgb.XGBClassifier(objective='binary:logistic', scale_pos_weight=class_weight, random_state=1, n_estimators=1000, early_stopping_rounds=10, enable_categorical=True, tree_method='hist', verbosity=0)
    # tmp_model.fit(x_tr_raw, y_tr, eval_set=[(x_va_raw, y_va)], verbose=False)

    # LGBM
    tmp_model = lgb.LGBMClassifier(class_weight='balanced', objective='binary', random_state=1, learning_rate= 0.01, verbosity=-1, n_estimators=1000)
    tmp_model.fit(x_tr_raw, y_tr, eval_set=[(x_va_raw, y_va)], eval_metric="binary_logloss", callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)])

    per_imp_results = permutation_importance(estimator=tmp_model, X=x_tr_raw, y=y_tr, n_repeats=10, scoring=profitability_scorer, random_state=1, n_jobs=-1)
    per_imp_df = pd.DataFrame({'feature': x_tr_raw.columns, 'importance':per_imp_results['importances_mean']})
    per_imp_features = per_imp_df.loc[per_imp_df['importance'] > 0, 'feature'].tolist()
    print(per_imp_features)
    top_features_list.append(per_imp_features)
    
    start_date_cv = val_end
    fold += 1

In [6]:
top_features_list

[['Feature_1',
  'Feature_2',
  'Feature_3',
  'Feature_6',
  'Feature_24',
  'Feature_46',
  'Feature_47',
  'Feature_86',
  'Feature_88',
  'Feature_113',
  'Feature_148',
  'Feature_157',
  'Feature_173',
  'Feature_178',
  'Feature_179',
  'Feature_180',
  'Feature_183',
  'Feature_188',
  'Feature_215',
  'Feature_216',
  'Feature_235',
  'Feature_237',
  'Feature_241',
  'Feature_282',
  'Feature_325',
  'Feature_404',
  'Feature_410',
  'Feature_417',
  'Feature_425',
  'Feature_472',
  'Feature_474',
  'Feature_478',
  'Feature_518',
  'Feature_521',
  'Feature_572',
  'Feature_582',
  'Feature_584',
  'Feature_587',
  'Feature_598',
  'Feature_604',
  'Feature_631',
  'Feature_640',
  'Feature_651',
  'Feature_652',
  'Feature_653',
  'Feature_654',
  'Feature_709',
  'Feature_711',
  'Feature_715'],
 ['Feature_1',
  'Feature_2',
  'Feature_3',
  'Feature_4',
  'Feature_15',
  'Feature_19',
  'Feature_68',
  'Feature_89',
  'Feature_101',
  'Feature_113',
  'Feature_157',
  'F